# Safe Speed ADB Challenge




![Workflow starting point](../assets/adp-safe-speed.png)


## Schema Reference

Extracted from `AI for Safer Roads 2026 - Data User Guide v1.0.pdf`.

| Field | Sample | Meaning |
|---|---:|---|
| `OBJECTID` | `9` | Unique ID created by GIS software |
| `english_ro` | `Surin Ring Road` | English street name |
| `OvertureID` | `9` | Unique ID created by GIS software |
| `SampleSize_avg` | `275650` | Average sample size if multiple TomTom datasets were collected |
| `RoadLength` | `3.7` | Ignore; use Shape_Length |
| `WeightedSample` | `1019905` | Average annual traffic weighted value by road length, obtained from TomTom sample size |
| `SampleSize Percent_` | `1.95E-05` | Ignore |
| `Percentile` | `0.04299693` | Travel percentile for this road; 0.042 maps to the 0-5 percent band |
| `SpeedLimit` | `90` | Speed limit obtained by TomTom, not validated |
| `RoadClass` | `primary` | Overture road class |
| `NumberOverLimit` | `96979` | Estimated annual vehicles over the speed limit based on the sample |
| `MedianSpeed` | `80.925` | Median, or 50th percentile, speed |
| `F85thPercentileSpeed` | `97.25` | 85th percentile speed |
| `ForAnalysis` | `90` | Ignore |
| `ProvinceID` | `32` | Province ID in Thailand only; not useful for this analysis |
| `SpeedLimitFloor` | `90` | Ignore |
| `PercentOverLimit` | `0.351819336` | Percentage estimate based on probe counts and speeds |
| `InvPercentile` | `0.95700307` | Inverse of traffic percentile, mainly for dashboard filtering |
| `AnalysisStatus` | `Valid` | Internal flag for analysis validation |
| `RankedPercentile` | `74.0577455` | Presentation ranking of roads by percentage traffic |
| `StreetImageLink` | `103.48332214,14.94148244,103.43913438,14.86699584` | Latitude/longitude values for Google Street View link generation |
| `LandUse` | `RURAL` | Land use category based on GRUMP data |
| `NO_OF_Result_Segments` | `4` | Ignore |
| `PercentileBand` | `0-5%` | Percentile band for dashboard |
| `SampleSizeTotal` | `275650` | Total sample size if multiple TomTom datasets were collected |
| `Shape_Length` | `11643.83585` | Geometric length of the shape |

In [55]:
# Core imports. These cells intentionally avoid hard dependencies on geospatial packages.
from pathlib import Path
import json
import math
import re
from statistics import median

import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

# Update this path after downloading challenge data.
GEOJSON_PATH = DATA_DIR / 'ADB_Innovation_Thailand.geojson'

GEOJSON_PATH


PosixPath('/Users/markjaysondoma/Documents/safe-speed-adb-challenge/data/ADB_Innovation_Thailand.geojson')

In [56]:
SCHEMA = {
    'OBJECTID': 'Unique ID created by GIS software',
    'english_ro': 'English street name',
    'OvertureID': 'Unique ID created by GIS software',
    'SampleSize_avg': 'Average sample size if multiple TomTom datasets were collected',
    'RoadLength': 'Ignore; use Shape_Length',
    'WeightedSample': 'Average annual traffic weighted value by road length, from TomTom sample size',
    'SampleSize Percent_': 'Ignore',
    'Percentile': 'Travel percentile for this road',
    'SpeedLimit': 'Speed limit obtained by TomTom, not validated',
    'RoadClass': 'Overture road class',
    'NumberOverLimit': 'Estimated annual vehicles over the speed limit based on the actual sample',
    'MedianSpeed': 'Median speed',
    'F85thPercentileSpeed': '85th percentile speed',
    'ForAnalysis': 'Ignore',
    'ProvinceID': 'Province ID in Thailand only; not useful for this analysis',
    'SpeedLimitFloor': 'Ignore',
    'PercentOverLimit': 'Percentage estimate based on probe counts and speeds',
    'InvPercentile': 'Inverse traffic percentile for dashboard filtering',
    'AnalysisStatus': 'Internal validation flag',
    'RankedPercentile': 'Presentation ranking of roads by percentage traffic',
    'StreetImageLink': 'Lat/lon values for street-view link generation',
    'LandUse': 'Urban/rural category based on GRUMP data',
    'NO_OF_Result_Segments': 'Ignore',
    'PercentileBand': 'Percentile band for dashboard',
    'SampleSizeTotal': 'Total sample size if multiple TomTom datasets were collected',
    'Shape_Length': 'Geometric length of the shape',
}

KEY_FIELDS = [
    'OBJECTID', 'english_ro', 'OvertureID', 'WeightedSample', 'Percentile', 'PercentileBand',
    'SpeedLimit', 'RoadClass', 'NumberOverLimit', 'MedianSpeed', 'F85thPercentileSpeed',
    'PercentOverLimit', 'LandUse', 'SampleSizeTotal', 'Shape_Length', 'StreetImageLink'
]

NUMERIC_FIELDS = [
    'SampleSize_avg', 'RoadLength', 'WeightedSample', 'Percentile', 'SpeedLimit',
    'NumberOverLimit', 'MedianSpeed', 'F85thPercentileSpeed', 'SpeedLimitFloor',
    'PercentOverLimit', 'InvPercentile', 'RankedPercentile', 'SampleSizeTotal', 'Shape_Length'
]


In [57]:
def load_geojson_features(path: Path) -> list[dict]:
    """Load features from a GeoJSON FeatureCollection or a single Feature."""
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)

    if data.get('type') == 'FeatureCollection':
        return data.get('features', [])
    if data.get('type') == 'Feature':
        return [data]
    raise ValueError(f'Unsupported GeoJSON root type: {data.get("type")}')


def feature_properties_frame(features: list[dict]) -> pd.DataFrame:
    rows = []
    for feature in features:
        props = dict(feature.get('properties') or {})
        geom = feature.get('geometry') or {}
        props['_geometry_type'] = geom.get('type')
        props['_coordinate_count'] = count_coordinates(geom.get('coordinates'))
        rows.append(props)
    return pd.DataFrame(rows)


def count_coordinates(coords) -> int:
    if coords is None:
        return 0
    if isinstance(coords, (list, tuple)) and coords and isinstance(coords[0], (int, float)):
        return 1
    if isinstance(coords, (list, tuple)):
        return sum(count_coordinates(item) for item in coords)
    return 0


def coerce_numeric_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')
    return out


def validate_schema(df: pd.DataFrame, expected_fields: list[str] = KEY_FIELDS) -> pd.DataFrame:
    rows = []
    for field in expected_fields:
        rows.append({
            'field': field,
            'present': field in df.columns,
            'missing_values': int(df[field].isna().sum()) if field in df.columns else None,
            'non_null_values': int(df[field].notna().sum()) if field in df.columns else 0,
            'description': SCHEMA.get(field, ''),
        })
    return pd.DataFrame(rows)


In [58]:
if GEOJSON_PATH.exists():
    features = load_geojson_features(GEOJSON_PATH)
    roads_raw = feature_properties_frame(features)
    roads_raw = coerce_numeric_columns(roads_raw, NUMERIC_FIELDS)
    roads = roads_raw.copy()
    print(f'Loaded {len(roads):,} road segments from {GEOJSON_PATH}')
else:
    features = []
    roads_raw = pd.DataFrame()
    roads = pd.DataFrame()
    print(f'Place the challenge GeoJSON at: {GEOJSON_PATH}')

roads.head()


Loaded 55,884 road segments from /Users/markjaysondoma/Documents/safe-speed-adb-challenge/data/ADB_Innovation_Thailand.geojson


,OBJECTID,english_ro,OvertureID,SampleSize_avg,RoadLength,WeightedSample,Percent_,Percentile,SpeedLimit,RoadClass,NumberOverLimit,MedianSpeed,F85thPercentileSpeed,ForAnalysis,ProvinceID,SpeedLimitFloor,PercentOverLimit,InvPercentile,AnalysisStatus,RankedPercentile,StreetImageLink,LandUse,NO_OF_Result_Segments,PercentileBand,SampleSizeTotal,Shape_Length,_geometry_type,_coordinate_count
0,1,Surin Ring Road,1,NaN,4.632086,NaN,NaN,NaN,NaN,primary,NaN,NaN,NaN,NaN,32,NaN,NaN,NaN,Not Included,NaN,"103.4699393,14.839138,103.43891053,14.86699584",NaN,NaN,NaN,NaN,4632.086492,LineString,52
1,2,Surin Ring Road,2,44479.6,2.300000,102303.08,0.000002,0.002591,66.0,primary,128017.0,78.38,115.2,66.0,32,60.0,0.575621,0.997409,Valid,26.253458,"103.43891053,14.86699584,103.48332214,14.94166589",RURAL,1.0,0-5%,222398.0,11672.918346,LineString,74
2,3,Surin Ring Road,3,NaN,0.122010,NaN,NaN,NaN,NaN,primary,NaN,NaN,NaN,NaN,32,NaN,NaN,NaN,Not Included,NaN,"103.48332214,14.94166589,103.48440114,14.94199564",NaN,NaN,NaN,NaN,122.010111,LineString,3
3,4,Surin Ring Road,4,NaN,2.045467,NaN,NaN,NaN,NaN,primary,NaN,NaN,NaN,NaN,32,NaN,NaN,NaN,Not Included,NaN,"103.48440114,14.94199564,103.50325276,14.94199566",NaN,NaN,NaN,NaN,2045.467119,LineString,17
4,5,Surin Ring Road,5,NaN,2.028143,NaN,NaN,NaN,NaN,primary,NaN,NaN,NaN,NaN,32,NaN,NaN,NaN,Not Included,NaN,"103.50325276,14.94199566,103.5216215,14.9380875",NaN,NaN,NaN,NaN,2028.142985,LineString,8


## GeoJSON EDA



In [59]:
def run_geojson_eda(df: pd.DataFrame, features: list[dict]) -> None:
    if df.empty:
        print('EDA will run after GEOJSON_PATH points to a real file.')
        return

    print(f'Rows / road segments: {len(df):,}')
    print(f'Columns: {len(df.columns):,}')
    print(f'Raw GeoJSON features: {len(features):,}')

    print('\nColumn overview')
    overview = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'non_null': df.notna().sum(),
        'missing': df.isna().sum(),
        'missing_pct': (df.isna().mean() * 100).round(2),
        'unique_values': df.nunique(dropna=True),
    }).sort_values(['missing_pct', 'unique_values'], ascending=[False, False])
    display(overview)

    if '_geometry_type' in df.columns:
        print('\nGeometry types')
        display(df['_geometry_type'].value_counts(dropna=False).rename_axis('geometry_type').reset_index(name='segments'))

    categorical_cols = [c for c in ['RoadClass', 'LandUse', 'PercentileBand', 'AnalysisStatus'] if c in df.columns]
    for col in categorical_cols:
        print(f'\nTop values: {col}')
        display(df[col].value_counts(dropna=False).head(20).rename_axis(col).reset_index(name='segments'))

    numeric_cols = [c for c in NUMERIC_FIELDS if c in df.columns]
    if numeric_cols:
        print('\nNumeric summary')
        display(df[numeric_cols].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T)

    id_cols = [c for c in ['OBJECTID', 'OvertureID'] if c in df.columns]
    for col in id_cols:
        duplicate_count = int(df[col].duplicated(keep=False).sum())
        print(f'Duplicate {col} rows: {duplicate_count:,}')

    speed_cols = [c for c in ['SpeedLimit', 'MedianSpeed', 'F85thPercentileSpeed', 'PercentOverLimit'] if c in df.columns]
    if speed_cols:
        print('\nSpeed field preview')
        display(df[speed_cols].head(10))


run_geojson_eda(roads, features)


Rows / road segments: 55,884
Columns: 28
Raw GeoJSON features: 55,884

Column overview


,dtype,non_null,missing,missing_pct,unique_values
RankedPercentile,float64,11544,44340,79.34,11544
Percentile,float64,11544,44340,79.34,11531
InvPercentile,float64,11544,44340,79.34,11531
WeightedSample,float64,11544,44340,79.34,11519
Percent_,float64,11544,44340,79.34,11519
SampleSize_avg,float64,11544,44340,79.34,11475
SampleSizeTotal,float64,11544,44340,79.34,11444
NumberOverLimit,float64,11544,44340,79.34,7613
PercentOverLimit,float64,11544,44340,79.34,7227
MedianSpeed,float64,11544,44340,79.34,2349



Geometry types


,geometry_type,segments
0,LineString,55884



Top values: RoadClass


...

,AnalysisStatus,segments
0,Not Included,44340
1,Valid,11544



Numeric summary


,count,mean,std,min,5%,25%,50%,75%,95%,max
SampleSize_avg,11544.0,6.697560e+05,2.976403e+06,3.333333,13679.275000,68135.287500,177238.000000,4.455252e+05,2.207437e+06,9.031018e+07
RoadLength,55884.0,1.374297e+00,4.605329e+00,0.000000,0.011500,0.029177,0.159897,7.866035e-01,6.204380e+00,1.466000e+02
WeightedSample,11544.0,4.538416e+06,3.892524e+07,0.000000,15859.620000,95435.237500,310401.700000,1.079965e+06,1.022300e+07,1.551703e+09
Percentile,11544.0,4.160893e-02,7.969394e-02,0.000000,0.000092,0.002308,0.012428,4.482255e-02,1.794565e-01,1.000000e+00
SpeedLimit,11596.0,8.053337e+01,1.843625e+01,0.000000,40.000000,80.000000,85.000000,9.000000e+01,9.000000e+01,1.200000e+02
NumberOverLimit,11544.0,2.114311e+05,1.271425e+06,0.000000,0.000000,0.000000,20466.500000,1.180442e+05,7.007855e+05,5.893935e+07
MedianSpeed,11544.0,6.454128e+01,2.123504e+01,0.000000,24.400000,53.718750,67.900000,7.940000e+01,9.250000e+01,1.300000e+02
F85thPercentileSpeed,11544.0,7.828160e+01,2.380317e+01,0.000000,37.000000,67.000000,82.000000,9.450000e+01,1.085000e+02,1.500000e+02
SpeedLimitFloor,11596.0,8.027596e+01,1.844006e+01,0.000000,40.000000,80.000000,80.000000,9.000000e+01,9.000000e+01,1.200000e+02
PercentOverLimit,11544.0,2.181039e-01,2.469572e-01,0.000000,0.000000,0.000000,0.125001,3.749988e-01,7.000002e-01,1.000000e+00


Duplicate OBJECTID rows: 0
Duplicate OvertureID rows: 0

Speed field preview


,SpeedLimit,MedianSpeed,F85thPercentileSpeed,PercentOverLimit
0,NaN,NaN,NaN,NaN
1,66.0,78.380000,115.200000,0.575621
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN
5,70.0,56.766667,84.666667,0.529277
6,30.0,15.450000,21.000000,0.300000
7,NaN,NaN,NaN,NaN
8,90.0,80.925000,97.250000,0.351819
9,NaN,NaN,NaN,NaN


## Filter Valid Analysis Rows

The EDA above describes the complete GeoJSON file. From this point onward, the notebook uses only rows where `AnalysisStatus == "Valid"`, because the speed and traffic analysis fields are populated for those segments.


In [60]:
if not roads_raw.empty and 'AnalysisStatus' in roads_raw.columns:
    roads = roads_raw[roads_raw['AnalysisStatus'].eq('Valid')].copy()
    print(f'Filtered working dataset to AnalysisStatus == Valid: {len(roads):,} of {len(roads_raw):,} road segments')
elif not roads_raw.empty:
    roads = roads_raw.copy()
    print('AnalysisStatus column not found; continuing with all loaded road segments')
else:
    roads = pd.DataFrame()
    print('Valid-row filter will run after the GeoJSON is loaded.')

roads.head()


Filtered working dataset to AnalysisStatus == Valid: 11,544 of 55,884 road segments


,OBJECTID,english_ro,OvertureID,SampleSize_avg,RoadLength,WeightedSample,Percent_,Percentile,SpeedLimit,RoadClass,NumberOverLimit,MedianSpeed,F85thPercentileSpeed,ForAnalysis,ProvinceID,SpeedLimitFloor,PercentOverLimit,InvPercentile,AnalysisStatus,RankedPercentile,StreetImageLink,LandUse,NO_OF_Result_Segments,PercentileBand,SampleSizeTotal,Shape_Length,_geometry_type,_coordinate_count
1,2,Surin Ring Road,2,44479.600000,2.3,1.023031e+05,1.952667e-06,2.591373e-03,66.0,primary,128017.0,78.380000,115.200000,66.0,32,60.0,0.575621,0.997409,Valid,26.253458,"103.43891053,14.86699584,103.48332214,14.94166589",RURAL,1.0,0-5%,222398.0,11672.918346,LineString,74
5,6,Surin Ring Road,6,27729.666667,1.9,5.268637e+04,1.005629e-06,9.392420e-04,70.0,primary,44030.0,56.766667,84.666667,70.0,32,70.0,0.529277,0.999061,Valid,16.459198,"103.5215639,14.9379571,103.50238738,14.94199565",URBAN,1.0,0-5%,83189.0,2116.242153,LineString,12
6,7,Surin Ring Road,7,3.333333,1.9,6.333333e+00,1.208848e-10,1.208848e-10,30.0,primary,3.0,15.450000,21.000000,30.0,32,30.0,0.300000,1.000000,Valid,0.129668,"103.50238738,14.94199565,103.48521899,14.94199565",RURAL,1.0,0-5%,10.0,1859.429082,LineString,16
8,9,Surin Ring Road,9,275650.000000,3.7,1.019905e+06,1.946700e-05,4.299693e-02,90.0,primary,96979.0,80.925000,97.250000,90.0,32,90.0,0.351819,0.957003,Valid,74.057746,"103.48332214,14.94148244,103.43913438,14.86699584",RURAL,4.0,0-5%,275650.0,11643.835851,LineString,82
17,18,Sukhaphiban 5 Road,18,744341.000000,0.2,1.488682e+05,2.841458e-06,4.771872e-03,80.0,secondary,66991.0,63.600000,76.000000,80.0,10,80.0,0.090000,0.995228,Valid,34.180498,"100.6520076,13.8817104,100.6554489,13.887161",URBAN,1.0,0-5%,744341.0,708.531851,LineString,6


In [61]:
if not roads.empty:
    display(validate_schema(roads))
else:
    print('Schema validation will run after GEOJSON_PATH points to a real file.')


,field,present,missing_values,non_null_values,description
0,OBJECTID,True,0,11544,Unique ID created by GIS software
1,english_ro,True,0,11544,English street name
2,OvertureID,True,0,11544,Unique ID created by GIS software
3,WeightedSample,True,0,11544,Average annual traffic weighted value by road ...
4,Percentile,True,0,11544,Travel percentile for this road
5,PercentileBand,True,0,11544,Percentile band for dashboard
6,SpeedLimit,True,0,11544,"Speed limit obtained by TomTom, not validated"
7,RoadClass,True,0,11544,Overture road class
8,NumberOverLimit,True,0,11544,Estimated annual vehicles over the speed limit...
9,MedianSpeed,True,0,11544,Median speed


## Speed Risk Metrics

 Gap between actual operating speed and posted speed.


In [62]:
def add_speed_risk_metrics(df: pd.DataFrame) -> pd.DataFrame:
    out = coerce_numeric_columns(df, NUMERIC_FIELDS)

    if {'F85thPercentileSpeed', 'SpeedLimit'}.issubset(out.columns):
        out['speed_gap_85th'] = out['F85thPercentileSpeed'] - out['SpeedLimit']
    else:
        out['speed_gap_85th'] = pd.NA

    if {'MedianSpeed', 'SpeedLimit'}.issubset(out.columns):
        out['speed_gap_median'] = out['MedianSpeed'] - out['SpeedLimit']
    else:
        out['speed_gap_median'] = pd.NA

    if {'PercentOverLimit', 'WeightedSample'}.issubset(out.columns):
        out['weighted_over_limit_pressure'] = out['PercentOverLimit'] * out['WeightedSample']
    else:
        out['weighted_over_limit_pressure'] = pd.NA

    if {'NumberOverLimit', 'Shape_Length'}.issubset(out.columns):
        km = out['Shape_Length'] / 1000
        out['over_limit_per_km'] = out['NumberOverLimit'] / km.replace(0, pd.NA)
    else:
        out['over_limit_per_km'] = pd.NA

    return out

roads_scored = add_speed_risk_metrics(roads) if not roads.empty else roads.copy()

risk_columns = [
    'OBJECTID', 'english_ro', 'RoadClass', 'LandUse', 'SpeedLimit', 'MedianSpeed',
    'F85thPercentileSpeed', 'speed_gap_85th', 'PercentOverLimit', 'WeightedSample',
    'weighted_over_limit_pressure', 'over_limit_per_km', 'PercentileBand'
]

if not roads_scored.empty:
    display(roads_scored[[c for c in risk_columns if c in roads_scored.columns]].head())
else:
    print('Risk metrics will populate after loading the GeoJSON.')


,OBJECTID,english_ro,RoadClass,LandUse,SpeedLimit,MedianSpeed,F85thPercentileSpeed,speed_gap_85th,PercentOverLimit,WeightedSample,weighted_over_limit_pressure,over_limit_per_km,PercentileBand
1,2,Surin Ring Road,primary,RURAL,66.0,78.380000,115.200000,49.200000,0.575621,1.023031e+05,58887.820000,10967.008952,0-5%
5,6,Surin Ring Road,primary,URBAN,70.0,56.766667,84.666667,14.666667,0.529277,5.268637e+04,27885.666667,20805.747556,0-5%
6,7,Surin Ring Road,primary,RURAL,30.0,15.450000,21.000000,-9.000000,0.300000,6.333333e+00,1.900000,1.613398,0-5%
8,9,Surin Ring Road,primary,RURAL,90.0,80.925000,97.250000,7.250000,0.351819,1.019905e+06,358822.300000,8328.784538,0-5%
17,18,Sukhaphiban 5 Road,secondary,URBAN,80.0,63.600000,76.000000,-4.000000,0.090000,1.488682e+05,13398.200000,94549.031076,0-5%


In [63]:
def summarize_speed_risk(df: pd.DataFrame) -> pd.DataFrame:
    group_cols = [col for col in ['RoadClass', 'LandUse'] if col in df.columns]
    if not group_cols:
        return pd.DataFrame()

    value_cols = {
        'OBJECTID': 'count',
        'Shape_Length': 'sum',
        'WeightedSample': 'sum',
        'PercentOverLimit': 'mean',
        'speed_gap_85th': 'median',
        'weighted_over_limit_pressure': 'sum',
        'over_limit_per_km': 'median',
    }
    value_cols = {k: v for k, v in value_cols.items() if k in df.columns}

    summary = df.groupby(group_cols, dropna=False).agg(value_cols).reset_index()
    summary = summary.rename(columns={
        'OBJECTID': 'segment_count',
        'Shape_Length': 'total_shape_length_m',
        'WeightedSample': 'total_weighted_sample',
        'PercentOverLimit': 'avg_percent_over_limit',
        'speed_gap_85th': 'median_85th_speed_gap',
        'weighted_over_limit_pressure': 'total_weighted_over_limit_pressure',
        'over_limit_per_km': 'median_over_limit_per_km',
    })
    return summary.sort_values(
        [c for c in ['total_weighted_over_limit_pressure', 'median_85th_speed_gap'] if c in summary.columns],
        ascending=False,
    )

if not roads_scored.empty:
    display(summarize_speed_risk(roads_scored))
else:
    print('Summary will run after data is loaded.')


,RoadClass,LandUse,segment_count,total_shape_length_m,total_weighted_sample,avg_percent_over_limit,median_85th_speed_gap,total_weighted_over_limit_pressure,median_over_limit_per_km
7,trunk,URBAN,962,5.359203e+06,1.774063e+10,0.344631,8.0,6.049338e+09,65532.813474
6,trunk,RURAL,1123,9.230296e+06,1.079335e+10,0.454561,15.0,5.337844e+09,58053.638411
1,motorway,URBAN,115,8.756161e+05,9.569434e+09,0.370483,6.5,3.266873e+09,289320.958728
3,primary,URBAN,1823,4.439156e+06,5.046106e+09,0.197074,-2.0,1.021237e+09,20879.808351
2,primary,RURAL,1489,7.894117e+06,2.888823e+09,0.354602,9.0,1.010103e+09,35575.091385
5,secondary,URBAN,2861,6.275834e+06,4.197751e+09,0.103917,-11.5,3.114265e+08,0.000000
4,secondary,RURAL,3144,2.639130e+07,1.387212e+09,0.141372,-5.0,2.649008e+08,810.155090
0,motorway,RURAL,27,1.515442e+05,7.681705e+08,0.152933,0.0,1.960397e+08,28193.468671


## Prioritize Candidate Segments

Note: Adjust weights


In [64]:
def percentile_rank(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors='coerce')
    # Keep missing values as NA so they contribute 0 to the final priority score.
    return numeric.rank(pct=True, na_option='keep')


def add_priority_score(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    components = []

    if 'weighted_over_limit_pressure' in out.columns:
        components.append(('pressure_rank', percentile_rank(out['weighted_over_limit_pressure']), 0.45))
    if 'speed_gap_85th' in out.columns:
        components.append(('speed_gap_rank', percentile_rank(out['speed_gap_85th']), 0.30))
    if 'PercentOverLimit' in out.columns:
        components.append(('percent_over_rank', percentile_rank(out['PercentOverLimit']), 0.15))
    if 'WeightedSample' in out.columns:
        components.append(('traffic_rank', percentile_rank(out['WeightedSample']), 0.10))

    score = pd.Series(0.0, index=out.index)
    total_weight = 0.0
    for name, values, weight in components:
        out[name] = values
        score = score + values.fillna(0) * weight
        total_weight += weight

    out['priority_score'] = score / total_weight if total_weight else pd.NA
    return out

prioritized = add_priority_score(roads_scored) if not roads_scored.empty else roads_scored.copy()

if not prioritized.empty:
    top_cols = [c for c in [
        'priority_score', 'OBJECTID', 'english_ro', 'RoadClass', 'LandUse', 'SpeedLimit',
        'F85thPercentileSpeed', 'speed_gap_85th', 'PercentOverLimit', 'WeightedSample',
        'weighted_over_limit_pressure', 'PercentileBand', 'StreetImageLink'
    ] if c in prioritized.columns]
    display(prioritized.sort_values('priority_score', ascending=False)[top_cols].head(25))
else:
    print('Priority ranking will run after data is loaded.')


,priority_score,OBJECTID,english_ro,RoadClass,LandUse,SpeedLimit,F85thPercentileSpeed,speed_gap_85th,PercentOverLimit,WeightedSample,weighted_over_limit_pressure,PercentileBand,StreetImageLink
15806,0.990120,15807,Udon Ratthaya Expressway,motorway,URBAN,80.0,115.000000,35.000000,0.854300,2.107923e+08,1.800800e+08,50-55%,"100.5432377,13.9018303,100.5565652,14.1450103"
16366,0.987110,16367,Prachim Ratthaya Expressway,motorway,URBAN,80.0,114.500000,34.500000,0.858595,5.437854e+07,4.668911e+07,35-40%,"100.4258304,13.799194,100.5495037,13.8097026"
16368,0.984893,16369,Prachim Ratthaya Expressway,motorway,URBAN,80.0,114.833333,34.833333,0.854856,3.158095e+07,2.699715e+07,25-30%,"100.5499176,13.8112534,100.427932,13.7984935"
177,0.984611,178,Buraphawithi Expressway,motorway,URBAN,80.0,109.947368,29.947368,0.727353,7.998745e+08,5.817913e+08,80-85%,"101.00488587,13.48893451,100.6639827,13.6596045"
36077,0.984031,36078,Phetkasem Road,trunk,RURAL,50.0,97.333333,47.333333,0.764385,2.544013e+07,1.944604e+07,25-30%,"98.4261369,8.3338412,98.50832367,8.42969965"
17518,0.983472,17519,Asian Highway,trunk,URBAN,80.0,112.333333,32.333333,0.761083,4.778082e+07,3.636516e+07,30-35%,"100.35200968,14.99199584,100.31325944,15.07532886"
35514,0.982662,35515,Chalong Rat Expressway,motorway,URBAN,80.0,108.666667,28.666667,0.714541,4.014564e+08,2.868570e+08,70-75%,"100.6890981,13.8974356,100.5940725,13.7050315"
15807,0.982298,15808,Udon Ratthaya Expressway,motorway,URBAN,80.0,106.800000,26.800000,0.822044,1.546902e+08,1.271621e+08,45-50%,"100.5569967,14.1452312,100.5439244,13.9048539"
32458,0.982136,32459,,motorway,URBAN,60.0,110.000000,50.000000,1.000000,1.066712e+07,1.066712e+07,15-20%,"100.6714758,13.655475,100.60495207,13.6749074"
17497,0.980219,17498,Asian Highway,trunk,URBAN,80.0,112.500000,32.500000,0.814579,2.082248e+07,1.696155e+07,20-25%,"100.31363238,15.07532886,100.35233777,14.99199584"


## Geometry And Attribute Checks

Note: Please review this section


In [65]:
if not roads_scored.empty:
    geometry_cols = [c for c in ['OBJECTID', 'english_ro', 'RoadClass', 'LandUse', 'Shape_Length', '_geometry_type', '_coordinate_count'] if c in roads_scored.columns]
    display(roads_scored[geometry_cols].describe(include='all'))

    if 'Shape_Length' in roads_scored.columns:
        display(
            roads_scored.sort_values('Shape_Length', ascending=False)[geometry_cols].head(10)
        )
else:
    print('Geometry QA will run after data is loaded.')


,OBJECTID,english_ro,RoadClass,LandUse,Shape_Length,_geometry_type,_coordinate_count
count,11544.000000,11544,11544,11544,11544.000000,11544,11544.000000
unique,NaN,685,4,2,NaN,1,NaN
top,NaN,,secondary,RURAL,NaN,LineString,NaN
freq,NaN,8020,6005,5783,NaN,11544,NaN
mean,33890.804401,NaN,NaN,NaN,5250.958374,NaN,65.266199
std,14402.511803,NaN,NaN,NaN,9178.055201,NaN,154.456117
min,2.000000,NaN,NaN,NaN,22.150957,NaN,2.000000
25%,21642.750000,NaN,NaN,NaN,783.552206,NaN,10.000000
50%,35775.500000,NaN,NaN,NaN,1821.366197,NaN,24.000000
75%,45340.500000,NaN,NaN,NaN,5385.550462,NaN,62.000000


,OBJECTID,english_ro,RoadClass,LandUse,Shape_Length,_geometry_type,_coordinate_count
30106,30107,,primary,RURAL,152218.413316,LineString,1225
17625,17626,,primary,RURAL,141024.289075,LineString,3325
37967,37968,,secondary,RURAL,127622.283366,LineString,3645
37637,37638,,secondary,RURAL,125773.687560,LineString,4837
37189,37190,,secondary,RURAL,124169.193871,LineString,2187
36662,36663,Chayangkun Road,primary,RURAL,103101.969011,LineString,666
17581,17582,Thanawithi Road,primary,RURAL,91151.039429,LineString,577
17642,17643,,primary,RURAL,88970.672164,LineString,1703
27520,27521,,primary,RURAL,87372.736703,LineString,1326
21940,21941,,primary,RURAL,84529.958115,LineString,1202


## Street Image And Mapillary Hooks



In [66]:
def parse_street_image_link(value):
    """Parse comma-separated lon/lat pairs from StreetImageLink-like values.

    The guide sample looks like: lon1,lat1,lon2,lat2.
    Returns a list of {'lon': ..., 'lat': ...} dictionaries.
    """
    if pd.isna(value):
        return []
    nums = [float(x) for x in re.findall(r'-?\d+(?:\.\d+)?', str(value))]
    pairs = []
    for i in range(0, len(nums) - 1, 2):
        pairs.append({'lon': nums[i], 'lat': nums[i + 1]})
    return pairs

if not prioritized.empty and 'StreetImageLink' in prioritized.columns:
    example = prioritized['StreetImageLink'].dropna().iloc[0] if prioritized['StreetImageLink'].notna().any() else None
    print('Example:', example)
    print('Parsed:', parse_street_image_link(example))
else:
    print('StreetImageLink parsing will run after data is loaded.')


Example: 103.43891053,14.86699584,103.48332214,14.94166589
Parsed: [{'lon': 103.43891053, 'lat': 14.86699584}, {'lon': 103.48332214, 'lat': 14.94166589}]


## Parse Street-Image Query Points

The Mapillary cache section uses these `query_lon` and `query_lat` columns as the coordinates for nearby image lookup.


In [67]:
def add_street_image_query_points(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if 'StreetImageLink' not in out.columns:
        out['query_lon'] = pd.NA
        out['query_lat'] = pd.NA
        out['street_image_point_count'] = 0
        return out

    parsed_points = out['StreetImageLink'].apply(parse_street_image_link)
    out['street_image_point_count'] = parsed_points.apply(len)
    out['query_lon'] = parsed_points.apply(lambda points: points[0]['lon'] if points else pd.NA)
    out['query_lat'] = parsed_points.apply(lambda points: points[0]['lat'] if points else pd.NA)
    return out


prioritized = add_street_image_query_points(prioritized) if not prioritized.empty else prioritized

if not prioritized.empty:
    display(prioritized[['OBJECTID', 'StreetImageLink', 'query_lon', 'query_lat', 'street_image_point_count']].head())
else:
    print('Street-image query points will be parsed after priority scoring runs.')


,OBJECTID,StreetImageLink,query_lon,query_lat,street_image_point_count
1,2,"103.43891053,14.86699584,103.48332214,14.94166589",103.438911,14.866996,2
5,6,"103.5215639,14.9379571,103.50238738,14.94199565",103.521564,14.937957,2
6,7,"103.50238738,14.94199565,103.48521899,14.94199565",103.502387,14.941996,2
8,9,"103.48332214,14.94148244,103.43913438,14.86699584",103.483322,14.941482,2
17,18,"100.6520076,13.8817104,100.6554489,13.887161",100.652008,13.881710,2


In [68]:
import os

try:
    import mapillary.interface as mly
except ModuleNotFoundError:
    mly = None
    print('Mapillary is not installed. Run `%pip install mapillary` or `pip install -r requirements.txt`, then restart the kernel.')

if mly is not None:
    token_candidates = [
        PROJECT_ROOT / 'MAPILLARY_TOKEN',
        Path.cwd() / 'MAPILLARY_TOKEN',
        Path.cwd().parent / 'MAPILLARY_TOKEN',
    ]
    token_path = next((path for path in token_candidates if path.exists()), None)
    MAPILLARY_TOKEN = os.environ.get('MAPILLARY_TOKEN')

    if not MAPILLARY_TOKEN and token_path is not None:
        MAPILLARY_TOKEN = token_path.read_text(encoding='utf-8').strip()

    if MAPILLARY_TOKEN:
        mly.set_access_token(MAPILLARY_TOKEN)
        print('Mapillary token loaded')
    else:
        print('Set MAPILLARY_TOKEN as an environment variable or create a MAPILLARY_TOKEN file in the project root.')


Mapillary token loaded


## Create Mapillary VRU Evidence Cache

Use this section to create `data/mapillary_vru_evidence.csv` from the Mapillary API. The CSV acts as a local cache: once it exists, later notebook runs can use it without calling Mapillary again.

By default this cell does not call the API. Set `BUILD_MAPILLARY_CACHE = True` when you want to fetch image metadata for the top-ranked segments.


In [69]:
VRU_EVIDENCE_PATH = DATA_DIR / 'mapillary_vru_evidence.csv'
BUILD_MAPILLARY_CACHE = False
OVERWRITE_MAPILLARY_CACHE = False
MAPILLARY_CACHE_TOP_N = 50
MAPILLARY_SEARCH_RADIUS_M = 100

VRU_REVIEW_COLUMNS = [
    'sidewalk_present',
    'crossing_present',
    'school_or_market_context',
    'transit_stop_context',
    'pedestrian_activity_visible',
    'speed_sign_visible',
    'notes',
]

# Make the dependency explicit: Mapillary lookups use query_lon/query_lat,
# which come from StreetImageLink via add_street_image_query_points().
if not prioritized.empty:
    prioritized = add_street_image_query_points(prioritized)


def mapillary_features_to_list(response) -> list[dict]:
    """Normalize SDK/API responses into a list of GeoJSON-like features."""
    if response is None:
        return []
    if hasattr(response, 'to_dict'):
        response = response.to_dict()
    if isinstance(response, dict):
        if isinstance(response.get('features'), list):
            return response['features']
        if response.get('type') == 'Feature':
            return [response]
    return []


def first_mapillary_image_metadata(lon: float, lat: float, radius_m: int = MAPILLARY_SEARCH_RADIUS_M) -> dict:
    """Fetch nearby Mapillary image metadata for one coordinate."""
    if mly is None:
        raise RuntimeError('Mapillary package is not available. Install mapillary and restart the kernel.')

    # The Mapillary SDK version in use fetches image vector tiles and accepts only
    # a small set of filters. Start with the most compatible radius-only call.
    try:
        response = mly.get_image_close_to(
            longitude=lon,
            latitude=lat,
            radius=radius_m,
        )
    except Exception:
        # Some SDK versions require an explicit image_type filter. Valid values are
        # 'pano', 'flat', and 'all'.
        response = mly.get_image_close_to(
            longitude=lon,
            latitude=lat,
            radius=radius_m,
            image_type='all',
        )
    features = mapillary_features_to_list(response)
    if not features:
        return {
            'mapillary_has_coverage': 0,
            'mapillary_image_count': 0,
        }

    feature = features[0]
    props = feature.get('properties') or {}
    geometry = feature.get('geometry') or {}
    coords = geometry.get('coordinates') or [pd.NA, pd.NA]
    image_id = props.get('id') or feature.get('id')

    thumbnail_url = pd.NA
    if image_id:
        try:
            thumbnail_url = mly.image_thumbnail(image_id=image_id, resolution=1024)
        except Exception:
            thumbnail_url = pd.NA

    return {
        'mapillary_has_coverage': 1,
        'mapillary_image_count': len(features),
        'mapillary_image_id': image_id,
        'mapillary_captured_at': props.get('captured_at'),
        'mapillary_compass_angle': props.get('compass_angle'),
        'mapillary_is_pano': props.get('is_pano'),
        'mapillary_image_lon': coords[0] if len(coords) > 0 else pd.NA,
        'mapillary_image_lat': coords[1] if len(coords) > 1 else pd.NA,
        'mapillary_thumbnail_url': thumbnail_url,
    }


def build_mapillary_vru_evidence_cache(df: pd.DataFrame, top_n: int = MAPILLARY_CACHE_TOP_N) -> pd.DataFrame:
    """Create a cache CSV with Mapillary image metadata plus blank review-label columns."""
    if df.empty:
        return pd.DataFrame()
    if 'OBJECTID' not in df.columns:
        raise ValueError('DataFrame must include OBJECTID')

    if 'query_lon' not in df.columns or 'query_lat' not in df.columns:
        df = add_street_image_query_points(df)

    ranked = df.sort_values('priority_score', ascending=False).head(top_n) if 'priority_score' in df.columns else df.head(top_n)
    rows = []

    for _, row in ranked.iterrows():
        query_lon = row.get('query_lon')
        query_lat = row.get('query_lat')
        base = {
            'OBJECTID': row.get('OBJECTID'),
            'english_ro': row.get('english_ro'),
            'RoadClass': row.get('RoadClass'),
            'LandUse': row.get('LandUse'),
            'priority_score': row.get('priority_score'),
            'query_lon': query_lon,
            'query_lat': query_lat,
        }

        if pd.notna(query_lon) and pd.notna(query_lat):
            try:
                base.update(first_mapillary_image_metadata(query_lon, query_lat))
            except Exception as exc:
                base.update({
                    'mapillary_has_coverage': 0,
                    'mapillary_image_count': 0,
                    'mapillary_error': str(exc),
                })
        else:
            base.update({
                'mapillary_has_coverage': 0,
                'mapillary_image_count': 0,
                'mapillary_error': 'No StreetImageLink coordinates',
            })

        for col in VRU_REVIEW_COLUMNS:
            base.setdefault(col, pd.NA)
        rows.append(base)

    cache = pd.DataFrame(rows)
    cache.to_csv(VRU_EVIDENCE_PATH, index=False)
    return cache


if VRU_EVIDENCE_PATH.exists() and not OVERWRITE_MAPILLARY_CACHE:
    print(f'Using existing Mapillary evidence cache: {VRU_EVIDENCE_PATH}')
    display(pd.read_csv(VRU_EVIDENCE_PATH).head())
elif BUILD_MAPILLARY_CACHE:
    cache = build_mapillary_vru_evidence_cache(prioritized, top_n=MAPILLARY_CACHE_TOP_N)
    print(f'Wrote Mapillary evidence cache: {VRU_EVIDENCE_PATH}')
    display(cache.head())
else:
    print('Mapillary cache not built. Set BUILD_MAPILLARY_CACHE = True to create data/mapillary_vru_evidence.csv from the API.')


Using existing Mapillary evidence cache: /Users/markjaysondoma/Documents/safe-speed-adb-challenge/data/mapillary_vru_evidence.csv


,OBJECTID,english_ro,RoadClass,LandUse,priority_score,query_lon,query_lat,mapillary_has_coverage,mapillary_image_count,mapillary_image_id,mapillary_captured_at,mapillary_compass_angle,mapillary_is_pano,mapillary_image_lon,mapillary_image_lat,mapillary_thumbnail_url,sidewalk_present,crossing_present,school_or_market_context,transit_stop_context,pedestrian_activity_visible,speed_sign_visible,notes
0,15807,Udon Ratthaya Expressway,motorway,URBAN,0.990120,100.543238,13.901830,1,481,8.113283e+14,1.543701e+12,16.442688,False,100.543249,13.901670,https://scontent.fmnl22-1.fna.fbcdn.net/m1/v/t...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,16367,Prachim Ratthaya Expressway,motorway,URBAN,0.987110,100.425830,13.799194,1,59,8.597180e+14,1.770984e+12,280.046056,False,100.426369,13.798532,https://scontent.fmnl22-1.fna.fbcdn.net/m1/v/t...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,16369,Prachim Ratthaya Expressway,motorway,URBAN,0.984893,100.549918,13.811253,1,3663,1.464357e+15,1.770751e+12,15.454982,False,100.549831,13.811014,https://scontent.fmnl22-1.fna.fbcdn.net/m1/v/t...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,178,Buraphawithi Expressway,motorway,URBAN,0.984611,101.004886,13.488935,1,526,3.950834e+15,1.540995e+12,201.462606,False,101.004717,13.488168,https://scontent.fmnl22-1.fna.fbcdn.net/m1/v/t...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,36078,Phetkasem Road,trunk,RURAL,0.984031,98.426137,8.333841,1,21,9.559503e+14,1.533943e+12,37.979584,False,98.425967,8.333625,https://scontent.fmnl22-1.fna.fbcdn.net/m1/v/t...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Vulnerable Road User Exposure

This section scores VRU context using challenge fields plus the optional `data/mapillary_vru_evidence.csv` cache. That cache can be created by the previous Mapillary API section, then reviewed by hand to fill columns such as `sidewalk_present`, `crossing_present`, `school_or_market_context`, and `pedestrian_activity_visible`.

If the cache does not exist, the notebook still runs using `LandUse`, `RoadClass`, traffic exposure, and street-image coordinate availability.

Note: ADD Computer Vision Model!!! sidewalk_present,crossing_present, school_or_market_context and pedestrian_activity_visible are currently blank in mapillary_cru_evidence.csv


In [72]:
VRU_EVIDENCE_PATH = DATA_DIR / 'mapillary_vru_evidence.csv'

VRU_BOOLEAN_COLUMNS = [
    'sidewalk_present',
    'crossing_present',
    'school_or_market_context',
    'transit_stop_context',
    'pedestrian_activity_visible',
    'speed_sign_visible',
]


def road_class_vru_weight(value) -> float:
    """Approximate VRU-relevance by road class before external POI data is joined."""
    road_class = str(value).lower()
    weights = {
        'secondary': 1.00,
        'primary': 0.85,
        'trunk': 0.60,
        'motorway': 0.10,
    }
    return weights.get(road_class, 0.50)


def normalize_boolean_series(series: pd.Series) -> pd.Series:
    true_values = {'1', 'true', 't', 'yes', 'y', 'present'}
    false_values = {'0', 'false', 'f', 'no', 'n', 'absent'}

    def normalize(value):
        if pd.isna(value):
            return pd.NA
        text = str(value).strip().lower()
        if text in true_values:
            return 1.0
        if text in false_values:
            return 0.0
        return pd.NA

    return series.map(normalize).astype('Float64')


def load_vru_evidence(path: Path = VRU_EVIDENCE_PATH) -> pd.DataFrame:
    """Load optional Mapillary API cache and reviewed VRU labels."""
    if not path.exists():
        return pd.DataFrame()

    evidence = pd.read_csv(path)
    if 'OBJECTID' not in evidence.columns:
        raise ValueError(f'{path} must contain an OBJECTID column')

    evidence = evidence.copy()
    evidence['OBJECTID'] = pd.to_numeric(evidence['OBJECTID'], errors='coerce').astype('Int64')
    for col in VRU_BOOLEAN_COLUMNS:
        if col in evidence.columns:
            evidence[col] = normalize_boolean_series(evidence[col])
    return evidence


def add_vru_exposure_features(df: pd.DataFrame, evidence_path: Path = VRU_EVIDENCE_PATH) -> pd.DataFrame:
    out = df.copy()

    # Make this cell safe to rerun by removing columns this section may have added earlier.
    vru_generated_cols = [
        'is_urban_land_use',
        'road_class_vru_weight',
        'has_street_image_coords',
        'has_mapillary_evidence',
        'has_manual_vru_evidence',
        'manual_vru_indicator_score',
        'vru_exposure_score',
        'safety_priority_score',
    ]
    cache_prefixes = ('mapillary_', 'query_')
    review_cols = VRU_BOOLEAN_COLUMNS + ['notes']
    columns_to_drop = [
        col for col in out.columns
        if col in vru_generated_cols or col in review_cols or col.startswith(cache_prefixes) or col.endswith('_evidence')
    ]
    if columns_to_drop:
        out = out.drop(columns=columns_to_drop)

    if 'LandUse' in out.columns:
        out['is_urban_land_use'] = out['LandUse'].astype(str).str.upper().eq('URBAN').astype(float)
    else:
        out['is_urban_land_use'] = 0.0

    if 'RoadClass' in out.columns:
        out['road_class_vru_weight'] = out['RoadClass'].map(road_class_vru_weight)
    else:
        out['road_class_vru_weight'] = 0.5

    if 'StreetImageLink' in out.columns:
        out['has_street_image_coords'] = out['StreetImageLink'].notna().astype(float)
    else:
        out['has_street_image_coords'] = 0.0

    evidence = load_vru_evidence(evidence_path)
    if not evidence.empty and 'OBJECTID' in out.columns:
        duplicate_context_cols = {'english_ro', 'RoadClass', 'LandUse', 'priority_score'}
        evidence_cols = ['OBJECTID'] + [
            c for c in evidence.columns
            if c != 'OBJECTID' and c not in duplicate_context_cols and c not in out.columns
        ]
        out = out.merge(evidence[evidence_cols], on='OBJECTID', how='left')
        out['has_mapillary_evidence'] = out.get('mapillary_has_coverage', 0).fillna(0).astype(float) if 'mapillary_has_coverage' in out.columns else 0.0
        out['has_manual_vru_evidence'] = 1.0
        out.loc[out[[c for c in VRU_BOOLEAN_COLUMNS if c in out.columns]].isna().all(axis=1), 'has_manual_vru_evidence'] = 0.0
    else:
        out['has_mapillary_evidence'] = 0.0
        out['has_manual_vru_evidence'] = 0.0

    present_evidence_cols = [c for c in VRU_BOOLEAN_COLUMNS if c in out.columns]
    if present_evidence_cols:
        # Positive VRU indicators: sidewalks/crossings imply pedestrian exposure, not necessarily safety.
        exposure_cols = [c for c in present_evidence_cols if c != 'speed_sign_visible']
        out['manual_vru_indicator_score'] = out[exposure_cols].mean(axis=1, skipna=True).fillna(0.0) if exposure_cols else 0.0
        out['speed_sign_visible'] = out['speed_sign_visible'].fillna(0.0) if 'speed_sign_visible' in out.columns else 0.0
    else:
        out['manual_vru_indicator_score'] = 0.0
        out['speed_sign_visible'] = 0.0

    traffic_component = percentile_rank(out['WeightedSample']) if 'WeightedSample' in out.columns else pd.Series(0.0, index=out.index)

    out['vru_exposure_score'] = (
        0.35 * out['is_urban_land_use'].fillna(0.0)
        + 0.20 * out['road_class_vru_weight'].fillna(0.0)
        + 0.15 * traffic_component.fillna(0.0)
        + 0.25 * out['manual_vru_indicator_score'].fillna(0.0)
        + 0.03 * out['has_street_image_coords'].fillna(0.0)
        + 0.02 * out.get('has_mapillary_evidence', 0).fillna(0.0)
    )

    if 'priority_score' in out.columns:
        out['safety_priority_score'] = (0.75 * out['priority_score'].fillna(0.0)) + (0.25 * out['vru_exposure_score'].fillna(0.0))

    return out


prioritized = add_vru_exposure_features(prioritized) if not prioritized.empty else prioritized

if not prioritized.empty:
    vru_cols = [c for c in [
        'safety_priority_score', 'priority_score', 'vru_exposure_score', 'OBJECTID', 'english_ro',
        'RoadClass', 'LandUse', 'is_urban_land_use', 'road_class_vru_weight',
        'manual_vru_indicator_score', 'has_manual_vru_evidence', 'has_mapillary_evidence', 'mapillary_image_count', 'has_street_image_coords'
    ] if c in prioritized.columns]
    display(prioritized.sort_values('safety_priority_score', ascending=False)[vru_cols].head(100))
else:
    print('VRU exposure scoring will run after data is loaded.')


,safety_priority_score,priority_score,vru_exposure_score,OBJECTID,english_ro,RoadClass,LandUse,is_urban_land_use,road_class_vru_weight,manual_vru_indicator_score,has_manual_vru_evidence,has_mapillary_evidence,mapillary_image_count,has_street_image_coords
4901,0.920402,0.980204,0.740995,34226,Srinagarindra-Rom Klao Road,secondary,URBAN,1.0,1.00,0.0,0.0,1.0,168.0,1.0
1222,0.909521,0.966836,0.737578,15760,Ratchaphruek Road,secondary,URBAN,1.0,1.00,0.0,0.0,1.0,748.0,1.0
2104,0.904587,0.983472,0.667934,17519,Asian Highway,trunk,URBAN,1.0,0.60,0.0,0.0,1.0,237.0,1.0
4653,0.904394,0.969952,0.707721,33833,Suranarai Road,primary,URBAN,1.0,0.85,0.0,0.0,1.0,199.0,1.0
2085,0.901625,0.980219,0.665842,17498,Asian Highway,trunk,URBAN,1.0,0.60,0.0,0.0,1.0,241.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4580,0.858566,0.912236,0.697557,33701,Mittraphap Road,primary,URBAN,1.0,0.85,0.0,0.0,0.0,NaN,1.0
6336,0.85854,0.934360,0.631081,36695,Highway 42,trunk,URBAN,1.0,0.60,0.0,0.0,0.0,NaN,1.0
6214,0.858391,0.918274,0.678742,36486,Phatthanakan Khu Khwang Road,primary,URBAN,1.0,0.85,0.0,0.0,0.0,NaN,1.0
5101,0.858219,0.929738,0.643659,34549,Sukhumvit Road,trunk,URBAN,1.0,0.60,0.0,0.0,0.0,NaN,1.0


## Export A First Triage Table

This produces a CSV you can share or use as the input to a mapping notebook/dashboard. The export keeps the original IDs plus the derived speed-risk and priority fields.


In [71]:
if not prioritized.empty:
    export_cols = [c for c in [
        'priority_score', 'OBJECTID', 'OvertureID', 'english_ro', 'RoadClass', 'LandUse',
        'SpeedLimit', 'MedianSpeed', 'F85thPercentileSpeed', 'speed_gap_median', 'speed_gap_85th',
        'PercentOverLimit', 'NumberOverLimit', 'WeightedSample', 'weighted_over_limit_pressure',
        'over_limit_per_km', 'Percentile', 'PercentileBand', 'SampleSizeTotal', 'Shape_Length',
        'StreetImageLink', 'vru_exposure_score', 'safety_priority_score', 'is_urban_land_use', 'road_class_vru_weight', 'manual_vru_indicator_score', 'has_manual_vru_evidence', 'has_mapillary_evidence', 'mapillary_image_count', 'mapillary_image_id', 'mapillary_thumbnail_url', 'has_street_image_coords'
    ] if c in prioritized.columns]

    out_path = OUTPUT_DIR / 'safe_speed_priority_segments.csv'
    prioritized.sort_values('priority_score', ascending=False)[export_cols].to_csv(out_path, index=False)
    print(f'Wrote {out_path}')
else:
    print('Export skipped until challenge data is loaded.')


Wrote /Users/markjaysondoma/Documents/safe-speed-adb-challenge/outputs/safe_speed_priority_segments.csv
